In [20]:
!uv add requests
!uv add openai
!uv add python-dotenv

Resolved 21 packages in 1ms
Checked 19 packages in 0.88ms
Resolved 21 packages in 0.77ms
Checked 19 packages in 0.88ms
Resolved 21 packages in 1ms
Checked 19 packages in 0.82ms


In [21]:
import os

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("KEYS")

In [22]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "openrouter/free"

In [23]:
cases = [
    {
        "id": 1,
        "ticket": "I was charged twice for my subscription this month.",
        "label": "billing",
    },
    {
        "id": 2,
        "ticket": "The app crashes whenever I try to upload a photo.",
        "label": "technical",
    },
    {
        "id": 3,
        "ticket": "I forgot my password and cannot log into my account.",
        "label": "account",
    },
    {
        "id": 4,
        "ticket": "I want to cancel my subscription immediately.",
        "label": "cancellation",
    },
    {
        "id": 5,
        "ticket": "Why did you charge me $49.99 when I expected $29.99?",
        "label": "billing",
    },
    {
        "id": 6,
        "ticket": "The website gives me a 500 error when I try to open my dashboard.",
        "label": "technical",
    },
    {
        "id": 7,
        "ticket": "I changed my email address but I cannot access my account anymore.",
        "label": "account",
    },
    {
        "id": 8,
        "ticket": "Please stop renewing my monthly plan.",
        "label": "cancellation",
    },
    {
        "id": 9,
        "ticket": "My credit card was declined when I tried to pay for the upgrade.",
        "label": "billing",
    },
    {
        "id": 10,
        "ticket": "The mobile application freezes every time I open the settings page.",
        "label": "technical",
    },
]

In [24]:
BASELINE_PROMPT = """
Classify the following customer-support ticket into exactly one
of these categories:

- billing
- technical
- account
- cancellation

Return ONLY the category name.

Ticket:
{ticket}
"""


In [25]:
IMPROVED_PROMPT = """
Classify customer-support tickets into exactly one of these categories:

- billing
- technical
- account
- cancellation

Use the following examples as guidance:

Example 1:
Ticket: "I was charged twice for the same subscription."
Category: billing

Example 2:
Ticket: "The application crashes whenever I upload a file."
Category: technical

Example 3:
Ticket: "I forgot my password and cannot sign in."
Category: account

Example 4:
Ticket: "I don't want my subscription to renew anymore."
Category: cancellation

Analyze the new ticket carefully before deciding.

Return ONLY one category name:
billing
technical
account
cancellation

Ticket:
{ticket}
"""


In [26]:
def ask_model(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
    )

    answer = response.choices[0].message.content.strip().lower()

    # Clean up occasional formatting
    answer = answer.replace("`", "").strip()

    return answer

In [27]:
results = []

for case in cases:
    print(f"Running case {case['id']}...")

    baseline = ask_model(
        BASELINE_PROMPT.format(ticket=case["ticket"])
    )

    improved = ask_model(
        IMPROVED_PROMPT.format(ticket=case["ticket"])
    )

    results.append({
        "id": case["id"],
        "ticket": case["ticket"],
        "correct_label": case["label"],
        "baseline_prediction": baseline,
        "improved_prediction": improved,
        "baseline_correct": baseline == case["label"],
        "improved_correct": improved == case["label"],
    })


Running case 1...


Running case 2...
Running case 3...
Running case 4...
Running case 5...
Running case 6...
Running case 7...
Running case 8...
Running case 9...
Running case 10...


In [28]:
baseline_correct = sum(
    r["baseline_correct"] for r in results
)

improved_correct = sum(
    r["improved_correct"] for r in results
)

total = len(results)

baseline_accuracy = baseline_correct / total
improved_accuracy = improved_correct / total

difference = improved_accuracy - baseline_accuracy

In [29]:
print("\n" + "=" * 50)
print("RESULTS")
print("=" * 50)

print(
    f"Baseline:  {baseline_correct}/{total} "
    f"= {baseline_accuracy:.0%}"
)

print(
    f"Improved:  {improved_correct}/{total} "
    f"= {improved_accuracy:.0%}"
)

print(
    f"Change:    {difference:+.0%} "
    f"percentage points"
)

print("\nIndividual results:")

for r in results:
    print(
        f"\nCase {r['id']}"
        f"\n  Correct:   {r['correct_label']}"
        f"\n  Baseline:  {r['baseline_prediction']}"
        f"\n  Improved:  {r['improved_prediction']}"
    )



RESULTS
Baseline:  10/10 = 100%
Improved:  10/10 = 100%
Change:    +0% percentage points

Individual results:

Case 1
  Correct:   billing
  Baseline:  billing
  Improved:  billing

Case 2
  Correct:   technical
  Baseline:  technical
  Improved:  technical

Case 3
  Correct:   account
  Baseline:  account
  Improved:  account

Case 4
  Correct:   cancellation
  Baseline:  cancellation
  Improved:  cancellation

Case 5
  Correct:   billing
  Baseline:  billing
  Improved:  billing

Case 6
  Correct:   technical
  Baseline:  technical
  Improved:  technical

Case 7
  Correct:   account
  Baseline:  account
  Improved:  account

Case 8
  Correct:   cancellation
  Baseline:  cancellation
  Improved:  cancellation

Case 9
  Correct:   billing
  Baseline:  billing
  Improved:  billing

Case 10
  Correct:   technical
  Baseline:  technical
  Improved:  technical


In [30]:
import csv

with open("results.csv", "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "id",
            "ticket",
            "correct_label",
            "baseline_prediction",
            "improved_prediction",
            "baseline_correct",
            "improved_correct",
        ],
    )

    writer.writeheader()
    writer.writerows(results)

print("\nSaved results to results.csv")


Saved results to results.csv
